# Notebook 03: a normal classifier or a one-class detector?

**Class 03_01.** If we *have* labelled attacks, why not train an ordinary (two-class) classifier, as for spam?
This notebook answers with three experiments on the KDD99 protocol of notebook 01:

1. **All attack kinds labelled:** logistic regression and a random forest trained on normal rows + the attack pool.
2. **Leave one category out:** the classifier never saw this kind of attack (a *new* attack, the "zero-day" case).
3. **Few labels:** only 10, 30 or 100 labelled attacks.

The one-class detectors of notebook 02 are the reference: they never saw any attack.

In [ ]:
import sys

import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

sys.path[:0] = [".", "../../src"]  # this folder and the generic libraries of the repository
import aasplot as ap
import anomlib as an
from aasplot import BLUE, GOLD, GREEN, GREY, RED

an.table_config()
ap.setup("03")
sp = an.kdd_split(seed=0)
Xtr, Xva, Xte, y = sp.X("train"), sp.X("val"), sp.X("test"), sp.y_test
Xp, cp = sp.X("pool"), sp.pool["category"].to_numpy()
cats = sp.test["category"]
print("labelled attack pool:", {c: int((cp == c).sum()) for c in an.CATEGORIES})


def supervised(make, keep: np.ndarray) -> dict:
    """Train on normal training rows (label 0) + the pool rows selected by `keep` (label 1); same threshold rule."""
    X = np.vstack([Xtr, Xp[keep]])
    yy = np.r_[np.zeros(len(Xtr)), np.ones(keep.sum())]
    m = make().fit(X, yy)
    s_va, s_te = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    return {**an.evaluate(s_te, y, s_va), **an.recall_by_category(s_te, cats, s_va)}


MODELS = {"logistic regression": lambda: LogisticRegression(max_iter=2000),
          "random forest": lambda: RandomForestClassifier(200, n_jobs=-1, random_state=0)}  # fmt: skip

## A. Every kind of attack labelled

In [ ]:
rows = [{"model": k, **supervised(f, np.ones(len(cp), bool))} for k, f in MODELS.items()]
for det in [an.GMM(), an.PCARecon()]:
    det.fit(Xtr)
    s_va, s_te = det.score(Xva), det.score(Xte)
    rows.append(
        {"model": det.name + " (one-class)", **an.evaluate(s_te, y, s_va), **an.recall_by_category(s_te, cats, s_va)}
    )
print(pl.DataFrame(rows))
rows_all = rows

In [ ]:
GMM = "GMM (-log-likelihood)"


def by(rows, key: str = "detector") -> dict:
    return {r[key]: r for r in rows}


PAL = [BLUE, RED, GOLD, GREEN]


def grouped(ax, groups, series, ylabel="", colors=PAL, width=0.8):
    n = len(series)
    for i, (name, vals) in enumerate(series):
        pos = np.arange(len(groups)) + (i - (n - 1) / 2) * width / n
        bars = ax.bar(pos, vals, width / n, color=colors[i], label=name, lw=0)
        for b, v in zip(bars, vals):
            ax.text(
                b.get_x() + b.get_width() / 2,
                v + 0.015,
                f"{v:.2f}".lstrip("0") if v < 1 else "1",
                ha="center",
                fontsize=6.5,
            )
    ax.set_xticks(np.arange(len(groups)), groups)
    ax.set(ylabel=ylabel, ylim=(0, 1.12))
    ax.grid(axis="x", visible=False)


def clf_all(rows) -> None:
    """rows: dicts with model, AUPRC, recall, r2l, u2r (logistic regression, random forest, one-class GMM)."""
    d = by(rows, "model")
    pick = [
        ("logistic regression", "logistic regression"),
        ("random forest", "random forest"),
        (f"{GMM} (one-class)", "GMM (one-class)"),
    ]
    ser = [(label, [d[k][m] for m in ("AUPRC", "recall", "r2l", "u2r")]) for k, label in pick]
    f, ax = ap.fig("full", 1.35)
    grouped(ax, ["AUPRC", "recall", "r2l recall", "u2r recall"], ser, "score")
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.2))
    ap.save(f, "clf_all")


clf_all(rows_all)

With labels for every kind of attack the random forest is almost perfect (AUPRC near 1): KDD99 is "too easy" for
supervised learning (Tavallaee et al. 2009). Precision stays near 0.63 only because the threshold allows 1% false
alarms; a supervised model can afford a much stricter one.

## B. Leave one category out (a new kind of attack)

In [ ]:
rows = []
for name, f in MODELS.items():
    for c in an.CATEGORIES:
        r = supervised(f, cp != c)
        rows.append({"model": name, "unseen": c, "recall on unseen": r[c], "AUPRC": r["AUPRC"]})
loco_df = pl.DataFrame(rows)
print(loco_df.pivot(on="unseen", index="model", values="recall on unseen"))
for det in [an.GMM(), an.PCARecon()]:
    det.fit(Xtr)
    rec = an.recall_by_category(det.score(Xte), cats, det.score(Xva))
    print("one-class (never saw any attack):", det.name, rec)
    if det.name.startswith("PCA"):
        rec_pca = rec  # the one-class reference of the plot below

In [ ]:
CATS = an.CATEGORIES


def loco(rows, one_class: dict) -> None:
    """rows: dicts with model, unseen, `recall on unseen`; one_class: recall per category of the PCA detector."""
    ser = []
    for model in dict.fromkeys(r["model"] for r in rows):
        v = {r["unseen"]: r["recall on unseen"] for r in rows if r["model"] == model}
        ser.append((model, [v[c] for c in CATS]))
    ser.append(("PCA (one-class, saw none)", [one_class[c] for c in CATS]))
    f, ax = ap.fig("full", 1.35)
    grouped(ax, CATS, ser, "recall on the unseen category", colors=[RED, BLUE, GREEN])
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.2))
    ap.save(f, "loco")


loco(loco_df.to_dicts(), rec_pca)

**The lesson of the module.** The random forest catches unseen dos, probe and u2r (they look like *some* attack it
knows), but its recall on **unseen r2l falls from 1.00 to 0.36**; logistic regression drops to 0.08. A classifier
learns the boundary between *normal* and *the attacks it saw*; an attack on the normal side of that boundary is
invisible. A one-class detector learns only *normal*: weaker on known attacks, but its recall does not depend on
having seen the attack (PCA: 0.45 on r2l, GMM 0.25, both better than logistic regression's 0.08; r2l is hard for
every method because it *is* close to normal). In practice both run side by side (signatures and classifiers for the known, anomaly detection for the
unknown).

## C. Few labels

In [ ]:
rng = np.random.default_rng(0)
rows = []
for n in [10, 30, 100, 300]:
    for rep in range(3):
        keep = np.zeros(len(cp), bool)
        keep[rng.choice(len(cp), n, replace=False)] = True
        rows.append({"labelled attacks": n, "rep": rep, **supervised(MODELS["random forest"], keep)})
few = pl.DataFrame(rows).group_by("labelled attacks").agg(pl.col(c).mean() for c in ["AUPRC", "recall", "r2l", "u2r"])
print(few.sort("labelled attacks"))

In [ ]:
def fewlabels(rows, gmm_auprc: float) -> None:
    """rows: dicts with `labelled attacks`, AUPRC, r2l, u2r (one per draw)."""
    ns = sorted({r["labelled attacks"] for r in rows})
    f, ax = ap.fig("full", 1.35)
    for m, lab, col, mk in (
        ("AUPRC", "AUPRC", BLUE, "o"),
        ("r2l", "recall r2l", RED, "s"),
        ("u2r", "recall u2r", GOLD, "^"),
    ):
        ax.plot(
            ns,
            [np.mean([r[m] for r in rows if r["labelled attacks"] == n]) for n in ns],
            color=col,
            marker=mk,
            ms=3.5,
            label=lab,
        )
    ax.axhline(gmm_auprc, color=GREY, ls="--", lw=0.9)
    ax.text(10.5, gmm_auprc - 0.1, f"GMM, no labels: AUPRC {gmm_auprc:.3f}", color=GREY, fontsize=7)
    ax.set(xscale="log", xlabel="labelled attacks (random forest)", ylabel="score", xticks=ns, ylim=(0, 1.05))
    ax.set_xticklabels([str(n) for n in ns])
    ax.legend(loc="lower right")
    ap.save(f, "fewlabels")


fewlabels(rows, float(next(r["AUPRC"] for r in rows_all if r["model"].startswith("GMM"))))

A random draw of labels is 95% DoS (that is what the pool holds), so with 10 or 30 labels the forest has seen almost
no r2l or u2r and misses them: with 10 labels its AUPRC (0.54) is below the one-class GMM (0.68), at 30 labels they tie,
from 100 labels on the forest wins. **Semi-supervised** methods
(Deep SAD, notebook 04's references) use both: a one-class model whose few labelled anomalies push the boundary.